In [1]:
# 1. Install uv
!pip install uv --quiet

# 2. Clone the repository
!git clone https://github.com/abdulrahman1238/clothes-segmentation-cvcy002

# 3. Change to the project directory
%cd /content/clothes-segmentation-cvcy002

# 4. Install project dependencies into Colab's system environment
# The --system flag is crucial for Colab compatibility.
# The -e . flag installs your local project in editable mode.
!uv pip install --system -e .

# 5. Install modelscope (required for your dataset code, but missing from pyproject.toml)
!uv pip install --system modelscope
!uv pip install --system addict
!uv pip install --system oss2

print("✅ Setup complete!")




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 95.1 MB/s eta 0:00:00
Cloning into 'clothes-segmentation-cvcy002'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 81 (delta 31), reused 66 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 175.60 KiB | 1004.00 KiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/clothes-segmentation-cvcy002
Using Python 3.13.15 environment at: /usr
Resolved 71 packages in 802ms
Prepared 22 packages in 53.37s
Uninstalled 20 packages in 944ms
Installed 22 packages in 384ms
 - cuda-toolkit==12.8.1
 + cuda-toolkit==12.6.3
 + cvcy002==0.1.0 (from file:///content/clothes-segmentation-cvcy002)
 - matplotlib==3.10.0
 + matplotlib==3.11.1
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.6.80
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12

In [2]:
# 6. Load the datasets
from modelscope.msdatasets import MsDataset
from modelscope.utils.constant import DownloadMode
dataset_val = MsDataset.load("LIP", split="validation", download_mode=DownloadMode.FORCE_REDOWNLOAD)
print(next(iter(dataset_val)))

2026-09-02 10:02:52,390 - modelscope - INFO - No subset_name specified, defaulting to the default
2026-09-02 10:02:56,675 - modelscope - INFO - Generating dataset dataset_builder (/root/.cache/modelscope/hub/datasets/modelscope/LIP/master/data_files)
2026-09-02 10:02:56,677 - modelscope - INFO - Loading meta-data file ...


  0%|          | 0/7378 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/oss2/api.py:703: SyntaxWarning: invalid escape sequence '\&'
  r'http://your-bucket.oss-cn-hangzhou.aliyuncs.com/logo.jpg?OSSAccessKeyId=YourAccessKeyId\&Expires=1447178011&Signature=UJfeJgvcypWq6Q%2Bm3IJcSHbvSak%3D'


100% {'Image:FILE': '/root/.cache/modelscope/hub/datasets/modelscope/LIP/master/data_files/extracted/b2830f17df7057467b81876cc1ca9fdf1358f94032f29747412880d3cb399008/val/val_images/100034_483681.jpg', 'Category_id:FILE': '/root/.cache/modelscope/hub/datasets/modelscope/LIP/master/data_files/extracted/b2830f17df7057467b81876cc1ca9fdf1358f94032f29747412880d3cb399008/val/val_segmentations/100034_483681.png'}


In [3]:
import os

# 1. Define the source (hidden cache) and destination (visible content) paths
source_dir = "/root/.cache/modelscope/hub/datasets/modelscope/LIP/master/data_files/extracted/b2830f17df7057467b81876cc1ca9fdf1358f94032f29747412880d3cb399008/val/"
dest_dir = "/content/LIP_dataset"

# 2. Check if the source exists and move it
if os.path.exists(source_dir):
    print(" Moving dataset... (this should be almost instant)")
    !mv {source_dir} {dest_dir}
    print(f" Successfully moved dataset to: {dest_dir}")
    print(" You can now see it in the left sidebar under 'content' -> 'LIP_dataset'")
else:
    print(" Source directory not found. Let's check what's actually in the cache:")
    !ls -la /root/.cache/modelscope/hub/datasets/modelscope/

 Moving dataset... (this should be almost instant)
 Successfully moved dataset to: /content/LIP_dataset
 You can now see it in the left sidebar under 'content' -> 'LIP_dataset'


In [4]:
import yaml
import os
import glob

new_paths = {
    "data_dir": '/content/LIP_dataset',
    "val_image_dir": "val_images",
    "val_mask_dir": "val_segmentations",
    "val_split_file": "val_id.txt"  # Adjust this later if the split file has a different name
}

# 3. Load, update, and save the config.yaml
config_path = "/content/clothes-segmentation-cvcy002/configs/config.yaml"

# Read the existing config
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

# Ensure the 'paths' key exists and update it
if 'paths' not in config:
    config['paths'] = {}
config['paths'].update(new_paths)

# Write the updated config back to the file
# sort_keys=False keeps your original file order intact
with open(config_path, 'w') as file:
    yaml.dump(config, file, sort_keys=False)

print("✅ config.yaml updated successfully!")
print("\n📝 New paths in config.yaml:")
for k, v in config['paths'].items():
    print(f"  {k}: {v}")

✅ config.yaml updated successfully!

📝 New paths in config.yaml:
  data_dir: /content/LIP_dataset
  train_image_dir: Train/images
  train_mask_dir: Train/segmentations
  train_split_file: Train/train_id.txt
  val_image_dir: val_images
  val_mask_dir: val_segmentations
  val_split_file: val_id.txt
  checkpoint_dir: outputs/checkpoints
  prediction_dir: outputs/predictions
  visualization_dir: outputs/visualizations
  metrics_dir: outputs/metrics


In [5]:
from google.colab import drive
drive.mount('/content/drive')


# Copy the file to your current directory
!cp "/content/drive/MyDrive/cvcy002/best_model.pth" /content/clothes-segmentation-cvcy002

Mounted at /content/drive


In [8]:
!MPLBACKEND=Agg uv run python scripts/evaluate.py --checkpoint /content/clothes-segmentation-cvcy002/best_model.pth

Using device: cuda
Building DeepLabV3+ | Backbone: resnet50 | Classes: 2 | Pretrained: True
 Loaded checkpoint from epoch 13
---------------------------------------------
Model Architecture: DeepLabV3PlusModel
Total Parameters:       26,677,842
Trainable Parameters:   26,677,842
---------------------------------------------
Starting Evaluation...
Visualizations saved to: outputs/visualizations/eval_visualizations.png
Metrics saved to: outputs/metrics/evaluation_results.json
----------------------------------------
Final mIoU:        0.8518
Pixel Accuracy:    0.9273
Macro F1-Score:    0.9194
Per Class IOU:    {'Non-clothes': 0.8951380252838135, 'Clothes': 0.8085182309150696}
Per Class F1-Score:    {'Non-clothes': 0.9446678161621094, 'Clothes': 0.8941222429275513}
----------------------------------------


In [ ]:
!MPLBACKEND=Agg uv run python scripts/evaluate.py --checkpoint outputs/checkpoints/best_model.pth